# FP&A Variance Report Generator — Stage 2: Data Validation

Goal: load raw input, catch bad data BEFORE calculating variance.
6 checks. Each check = one cell, one output, one reason logged.

In [1]:
import pandas as pd

sheets = pd.read_excel("sample_input.xlsx", sheet_name=None)
actuals = sheets["Actuals"]
budget = sheets["Budget"]
prior = sheets["Prior_Month"]
department = sheets["Department"]
control = sheets["Control_Totals"]

actuals.shape, actuals.columns.tolist()

((110, 5),
 ['month', 'cost_center', 'department', 'account_category', 'actual_amount'])

## Exceptions log

One table. Every issue any check finds gets a row here: which sheet, which row, what's wrong.
This table = your "exceptions requiring human review" deliverable.

In [2]:
exceptions = []

def flag(sheet, row_idx, issue, detail):
    exceptions.append({"sheet": sheet, "row": row_idx, "issue": issue, "detail": detail})

## Check 1 — missing cost center

In [3]:
missing_cc = actuals[actuals["cost_center"].isna() | (actuals["cost_center"] == "")]
for idx in missing_cc.index:
    flag("Actuals", idx, "missing_cost_center", "cost_center blank")

missing_cc

,month,cost_center,department,account_category,actual_amount
36,2026-06-01,NaN,Manufacturing,Revenue,4700079.07
37,2026-06-01,NaN,Manufacturing,Raw Material Cost,1618237.00


## Check 2 — missing account category

In [4]:
missing_cat = actuals[actuals["account_category"].isna() | (actuals["account_category"] == "")]
for idx in missing_cat.index:
    flag("Actuals", idx, "missing_account_category", "account_category blank")

missing_cat

,month,cost_center,department,account_category,actual_amount
72,2026-08-01,CC-100,Manufacturing,NaN,4837317.64
73,2026-08-01,CC-100,Manufacturing,NaN,1787369.81


## Check 3 — duplicate records
Same month + cost_center + account_category appearing twice = duplicate entry.

In [5]:
dup_mask = actuals.duplicated(subset=["month","cost_center","account_category"], keep="first")
dupes = actuals[dup_mask]
for idx in dupes.index:
    flag("Actuals", idx, "duplicate_record", "same month/cost_center/category seen earlier")

dupes

,month,cost_center,department,account_category,actual_amount
73,2026-08-01,CC-100,Manufacturing,NaN,1787369.81
108,2026-04-01,CC-100,Manufacturing,Admin Overheads,253841.32
109,2026-05-01,CC-100,Manufacturing,Employee Cost,958343.38


## Check 4 — invalid dates
`pd.to_datetime(..., errors="coerce")` turns anything unparseable into NaT (Not a Time).

In [6]:
parsed_dates = pd.to_datetime(actuals["month"], errors="coerce")
bad_dates = actuals[parsed_dates.isna()]
for idx in bad_dates.index:
    flag("Actuals", idx, "invalid_date", f"unparseable month value: {actuals.loc[idx,'month']}")

bad_dates

,month,cost_center,department,account_category,actual_amount
54,2026-13-45,CC-100,Manufacturing,Revenue,4870979.02


## Check 5 — unexpected negative values
Revenue should never be negative. (Cost categories CAN legitimately be negative — e.g. a credit/refund — so we only flag Revenue here.)

In [7]:
neg_revenue = actuals[(actuals["account_category"] == "Revenue") & (actuals["actual_amount"] < 0)]
for idx in neg_revenue.index:
    flag("Actuals", idx, "unexpected_negative", f"Revenue negative: {actuals.loc[idx,'actual_amount']}")

neg_revenue

,month,cost_center,department,account_category,actual_amount
90,2026-09-01,CC-100,Manufacturing,Revenue,-5148111.05


## Check 6 — control total mismatch
Sum of Actuals per month should equal Control_Totals. Mismatch = something missing/duplicated upstream, or a genuine data entry error.

In [8]:
# exclude rows already flagged as bad data (blank cc/category, bad dates) from the sum,
# so one bad row doesn't mask or fake a control-total match
clean_actuals = actuals.drop(index=set(missing_cc.index) | set(missing_cat.index) | set(bad_dates.index) | set(dupes.index))

monthly_actual = clean_actuals.groupby("month")["actual_amount"].sum().reset_index()
monthly_actual.columns = ["month", "sum_of_actuals"]

check = control.merge(monthly_actual, on="month", how="left")
check["diff"] = (check["control_total_actual"] - check["sum_of_actuals"]).round(2)
mismatches = check[check["diff"].abs() > 1]  # tolerance ₹1 for rounding

for _, r in mismatches.iterrows():
    flag("Control_Totals", r["month"], "control_total_mismatch", f"control={r['control_total_actual']}, sum_actuals={r['sum_of_actuals']}, diff={r['diff']}")

check

,month,control_total_actual,sum_of_actuals,diff
0,2026-04-01,18390817.12,18390817.12,0.00
1,2026-05-01,18779869.18,18779869.18,0.00
2,2026-06-01,18034424.48,11716108.41,6318316.07
3,2026-07-01,18859977.77,13988998.75,4870979.02
4,2026-08-01,18869587.37,12194899.92,6674687.45
5,2026-09-01,19445508.33,9149286.23,10296222.10


## All exceptions — the review log

In [9]:
exceptions_df = pd.DataFrame(exceptions)
print(f"Total exceptions found: {len(exceptions_df)}")
exceptions_df

Total exceptions found: 13


,sheet,row,issue,detail
0,Actuals,36,missing_cost_center,cost_center blank
1,Actuals,37,missing_cost_center,cost_center blank
2,Actuals,72,missing_account_category,account_category blank
3,Actuals,73,missing_account_category,account_category blank
4,Actuals,73,duplicate_record,same month/cost_center/category seen earlier
5,Actuals,108,duplicate_record,same month/cost_center/category seen earlier
6,Actuals,109,duplicate_record,same month/cost_center/category seen earlier
7,Actuals,54,invalid_date,unparseable month value: 2026-13-45
8,Actuals,90,unexpected_negative,Revenue negative: -5148111.05
9,Control_Totals,2026-06-01,control_total_mismatch,"control=18034424.48, sum_actuals=11716108.41, ..."


## Clean dataset for Stage 3

`clean_actuals` = actuals with the row-level bad data removed (blank cc/category, bad dates, duplicates).
Control-total mismatches are logged but NOT auto-corrected — that needs a human, which is exactly the point.
This clean set feeds the variance calculations next.

In [10]:
print("Original rows:", len(actuals))
print("Clean rows after validation:", len(clean_actuals))
print("Exceptions logged:", len(exceptions_df))

clean_actuals.head()

Original rows: 110
Clean rows after validation: 103
Exceptions logged: 13


,month,cost_center,department,account_category,actual_amount
0,2026-04-01,CC-100,Manufacturing,Revenue,4723064.29
1,2026-04-01,CC-100,Manufacturing,Raw Material Cost,1712428.40
2,2026-04-01,CC-100,Manufacturing,Employee Cost,923319.50
3,2026-04-01,CC-100,Manufacturing,Marketing Expense,406668.13
4,2026-04-01,CC-100,Manufacturing,Logistics Cost,315239.26
